In [ ]:

# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ASCEND EchoLV — Application Launcher
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

APP_DIR = Path("/content/drive/MyDrive/EchoLV_Deployment")

if not APP_DIR.exists():
    raise FileNotFoundError(
        f"Application folder not found: {APP_DIR}"
    )

required_files = [
    "app.py",
    "echolv_app.py",
    "frozen_inference.py",
    "phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras",
    "phase3_classification/phase3C_classifier_development/models/phase3C_selected_logistic_regression.joblib",
]

for file in required_files:
    path = APP_DIR / file

    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")

    print("✓", file)

print("\n✓ Google Drive mounted successfully")
print("✓ ASCEND EchoLV application files found")


Mounted at /content/drive
✓ app.py
✓ echolv_app.py
✓ frozen_inference.py
✓ phase1_segmentation/phase1F_final_model/models/final_lv_segmentation_model.keras
✓ phase3_classification/phase3C_classifier_development/models/phase3C_selected_logistic_regression.joblib

✓ Google Drive mounted successfully
✓ ASCEND EchoLV application files found


In [ ]:

# ============================================================
# CELL 2 — INSTALL APPLICATION DEPENDENCIES
# ============================================================

%pip install -q \
    "gradio" \
    "tensorflow==2.21.0" \
    "numpy" \
    "pandas" \
    "scipy" \
    "scikit-learn==1.6.1" \
    "joblib" \
    "nibabel" \
    "opencv-python-headless" \
    "Pillow" \
    "matplotlib" \
    "reportlab" \
    "psutil"

print("\n✓ ASCEND EchoLV dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.4 MB/s eta 0:00:00

✓ ASCEND EchoLV dependencies installed


In [ ]:

# ============================================================
# CELL 3 — LAUNCH ASCEND ECHOLV WITH ORIGINAL UI
# ============================================================

import sys
import importlib.util
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EchoLV_Deployment")

sys.path.insert(0, str(APP_DIR))

def load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(
        module_name,
        str(file_path)
    )

    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {file_path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    return module


print("=" * 60)
print("ASCEND EchoLV — Starting Application")
print("=" * 60)

# ------------------------------------------------------------
# STEP 1 — Load the frozen inference pipeline
# ------------------------------------------------------------

print("\n[1/4] Loading frozen segmentation and classifier...")

frozen = load_module(
    "ascend_frozen_inference",
    APP_DIR / "frozen_inference.py"
)

print("✓ Frozen inference pipeline loaded")


# ------------------------------------------------------------
# STEP 2 — Load the original Gradio application
# ------------------------------------------------------------

print("\n[2/4] Loading ASCEND EchoLV interface...")

echolv = load_module(
    "echolv_app",
    APP_DIR / "echolv_app.py"
)

print("✓ Original application module loaded")


# ------------------------------------------------------------
# STEP 3 — Build the application
# ------------------------------------------------------------

print("\n[3/4] Building the complete application...")

records_dir = APP_DIR / "records"
records_dir.mkdir(parents=True, exist_ok=True)

demo, launch_kwargs = echolv.build_app(
    frozen=frozen,
    records_dir=records_dir,
    demo_paths=None
)

print("✓ Application built successfully")

# Verify original styling
print("\nOriginal styling configuration:")
print("Custom CSS:", "✓" if launch_kwargs.get("css") else "Missing")
print("Custom theme:", "✓" if launch_kwargs.get("theme") else "Missing")
print("Custom head:", "✓" if launch_kwargs.get("head") else "Missing")


# ------------------------------------------------------------
# STEP 4 — Launch with the ORIGINAL THEME AND CSS
# ------------------------------------------------------------

print("\n[4/4] Launching ASCEND EchoLV...")
print("✓ Applying original CSS and theme")
print("✓ Generating temporary public link\n")

demo.launch(
    share=True,
    debug=True,
    show_error=True,
    **launch_kwargs
)


ASCEND EchoLV — Starting Application

[1/4] Loading frozen segmentation and classifier...
FROZEN PHASE 2 MEASUREMENT FUNCTIONS

✓ 256×256 stretch preprocessing
✓ Post-resize per-image min-max normalization
✓ Anisotropic effective spacing
✓ Threshold >= 0.50
✓ 8-connected largest component
✓ Binary hole filling
✓ Physical LV area
✓ Exact PCA major-axis length
✓ Single-plane area-length volume
FROZEN SEGMENTATION MODEL LOADED

Parameters: 7,759,521
Input shape : (None, 256, 256, 1)
Output shape: (None, 256, 256, 1)

✓ Frozen Phase 1 model ready for inference.
✓ Single-frame frozen inference function ready.
COMPLETE FROZEN A4C PIPELINE

ED/ES input
   ↓
Frozen LV segmentation
   ↓
Physical LV geometry
   ↓
EDV / ESV / SV / EF
   ↓
Frozen EF-band classifier
   ↓
EF band + uncalibrated confidence

✓ analyze_a4c_pair() ready.
✓ Frozen inference pipeline loaded

[2/4] Loading ASCEND EchoLV interface...
✓ Original application module loaded

[3/4] Building the complete application...
✓ Applicat

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://908aa03d4f2728c63e.gradio.live
